In [2]:
import pandas as pd
import random
from faker import Faker
from sqlalchemy import create_engine ,text
from sqlalchemy.engine import URL

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

def get_engine():
    url = URL.create(
        drivername="mysql+pymysql",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"),
        port=3306,
        database="Banking_analytics"
    )
    engine = create_engine(url)

    return engine

In [27]:
from sqlalchemy import text

query = """
CREATE TABLE loans (
    loan_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT NOT NULL,
    loan_type VARCHAR(30) NOT NULL,
    loan_amount DECIMAL(12,2) NOT NULL,
    interest_rate DECIMAL(4,2) NOT NULL,
    tenure_months INT NOT NULL,
    emi DECIMAL(12,2) NOT NULL,
    loan_status VARCHAR(20) NOT NULL,
    loan_start_date DATE NOT NULL,

    FOREIGN KEY (customer_id)
    REFERENCES customers(customer_id)
);
"""

with engine.connect() as conn:
    conn.execute(text(query))
    conn.commit()

print("Loans table created successfully!")

Loans table created successfully!


In [28]:
pd.read_sql("DESCRIBE loans;", engine)


,Field,Type,Null,Key,Default,Extra
0,loan_id,int,NO,PRI,None,auto_increment
1,customer_id,int,NO,MUL,None,
2,loan_type,varchar(30),NO,,None,
3,loan_amount,"decimal(12,2)",NO,,None,
4,interest_rate,"decimal(4,2)",NO,,None,
5,tenure_months,int,NO,,None,
6,emi,"decimal(12,2)",NO,,None,
7,loan_status,varchar(20),NO,,None,
8,loan_start_date,date,NO,,None,


In [29]:
query = """
SELECT
    c.customer_id,
    c.age,
    c.gender,
    c.marital_status,
    c.occupation,
    c.annual_income,
    c.cibil_score,
    a.account_type,
    a.account_balance,
    a.account_opening_date
FROM customers c
INNER JOIN accounts a
ON c.customer_id = a.customer_id;
"""

loan_df = pd.read_sql(query, engine)

In [31]:
loan_df.head()



,customer_id,age,gender,marital_status,occupation,annual_income,cibil_score,account_type,account_balance,account_opening_date
0,1,43,Female,Single,Equipment,1125000,391,Saving,3860000.0,2023-01-11
1,2,75,Female,Married,Prosthodontist,367500,782,Saving,18500.0,1976-09-05
2,3,60,Male,Married,Social Work Teacher,158000,531,Saving,133000.0,2002-04-11
3,4,23,Male,Single,Farmworker,2055000,815,Saving,3000.0,2021-06-17
4,5,70,Male,Married,Secondary School Teacher,2300000,785,Saving,83000.0,1978-10-31


In [34]:
import random
loan_types = [
    "Home Loan",
    "Personal Loan",
    "Car Loan",
    "Education Loan",
    "Business Loan"
]

loan_df["loan_type"] = random.choices(
    loan_types,
    weights=[25, 35, 20, 10, 10],
    k=len(loan_df)
)
loan_df["loan_type"].value_counts()

loan_type
Personal Loan     34778
Home Loan         25059
Car Loan          20043
Education Loan    10212
Business Loan      9908
Name: count, dtype: int64

In [35]:
def generate_loan_amount(loan_type):
    
    if loan_type == "Personal Loan":
        return random.randint(50000, 1000000)

    elif loan_type == "Home Loan":
        return random.randint(1000000, 10000000)

    elif loan_type == "Car Loan":
        return random.randint(300000, 2000000)

    elif loan_type == "Education Loan":
        return random.randint(100000, 3000000)

    else:   # Business Loan
        return random.randint(500000, 5000000)

In [36]:
loan_df["loan_amount"] = loan_df["loan_type"].apply(generate_loan_amount)

In [37]:
loan_df[["loan_type", "loan_amount"]].head(10)

,loan_type,loan_amount
0,Personal Loan,844353
1,Personal Loan,591923
2,Home Loan,5268589
3,Home Loan,9700247
4,Personal Loan,488229
5,Business Loan,4699251
6,Home Loan,1971972
7,Personal Loan,275770
8,Business Loan,4347976
9,Personal Loan,70483


In [38]:
def generate_interest_rate(cibil):

    if cibil >= 800:
        return round(random.uniform(8.0, 9.0), 2)

    elif cibil >= 750:
        return round(random.uniform(9.0, 10.5), 2)

    elif cibil >= 700:
        return round(random.uniform(10.5, 12.0), 2)

    elif cibil >= 650:
        return round(random.uniform(12.0, 14.0), 2)

    else:
        return round(random.uniform(14.0, 18.0), 2)

In [39]:
loan_df["interest_rate"] = loan_df["cibil_score"].apply(generate_interest_rate)

In [40]:
loan_df[["cibil_score", "interest_rate"]].head(10)

,cibil_score,interest_rate
0,391,14.94
1,782,9.46
2,531,15.56
3,815,8.82
4,785,9.70
5,757,9.59
6,646,17.06
7,729,11.62
8,685,13.42
9,733,11.35


In [41]:
def generate_tenure(loan_type):

    if loan_type == "Personal Loan":
        return random.choice([12, 24, 36, 48, 60])

    elif loan_type == "Home Loan":
        return random.choice([120, 180, 240, 300, 360])

    elif loan_type == "Car Loan":
        return random.choice([36, 48, 60, 72, 84])

    elif loan_type == "Education Loan":
        return random.choice([24, 36, 48, 60, 84])

    else:   # Business Loan
        return random.choice([36, 60, 84, 120])

In [42]:
loan_df["tenure_months"] = loan_df["loan_type"].apply(generate_tenure)

In [43]:
loan_df[["loan_type", "tenure_months"]].head(10)

,loan_type,tenure_months
0,Personal Loan,12
1,Personal Loan,36
2,Home Loan,360
3,Home Loan,180
4,Personal Loan,48
5,Business Loan,120
6,Home Loan,360
7,Personal Loan,48
8,Business Loan,84
9,Personal Loan,60


In [44]:
def calculate_emi(loan_amount, interest_rate, tenure_months):
    
    monthly_rate = interest_rate / (12 * 100)

    emi = (
        loan_amount
        * monthly_rate
        * (1 + monthly_rate) ** tenure_months
    ) / (
        (1 + monthly_rate) ** tenure_months - 1
    )

    return round(emi, 2)

In [45]:
loan_df["emi"] = loan_df.apply(
    lambda row: calculate_emi(
        row["loan_amount"],
        row["interest_rate"],
        row["tenure_months"]
    ),
    axis=1
)

In [46]:
loan_df[
    [
        "loan_amount",
        "interest_rate",
        "tenure_months",
        "emi"
    ]
].head(10)

,loan_amount,interest_rate,tenure_months,emi
0,844353,14.94,12,76185.97
1,591923,9.46,36,18949.97
2,5268589,15.56,360,68983.60
3,9700247,8.82,180,97350.39
4,488229,9.70,48,12312.53
5,4699251,9.59,120,61038.96
6,1971972,17.06,360,28209.99
7,275770,11.62,48,7210.74
8,4347976,13.42,84,80094.40
9,70483,11.35,60,1544.80


In [47]:
def generate_loan_status(row):

    cibil = row["cibil_score"]
    income = row["annual_income"]
    loan = row["loan_amount"]

    # Poor credit score
    if cibil < 550:
        return "Rejected"

    # Loan amount too high compared to income
    if income < 300000 and loan > income * 10:
        return "Rejected"

    # Strong applicant
    if cibil >= 750 and income >= 500000:
        return "Approved"

    # Average applicant
    if 650 <= cibil < 750:
        return random.choices(
            ["Approved", "Pending"],
            weights=[80, 20],
            k=1
        )[0]

    # Remaining applicants
    return "Pending"

In [48]:
loan_df["loan_status"] = loan_df.apply(generate_loan_status, axis=1)

In [49]:
loan_df["loan_status"].value_counts()

loan_status
Approved    41149
Pending     39321
Rejected    19530
Name: count, dtype: int64

In [50]:
from datetime import datetime, timedelta
def generate_loan_start_date():

    start_date = datetime(2018, 1, 1)
    end_date = datetime(2025, 12, 31)

    random_days = random.randint(
        0,
        (end_date - start_date).days
    )

    return (
        start_date + timedelta(days=random_days)
    ).date()

In [51]:
loan_df["loan_start_date"] = [
    generate_loan_start_date()
    for _ in range(len(loan_df))
]

In [52]:
loan_df[
    [
        "loan_start_date",
        "loan_status"
    ]
].head(10)

,loan_start_date,loan_status
0,2023-04-16,Rejected
1,2020-12-21,Pending
2,2019-06-03,Rejected
3,2022-11-07,Approved
4,2018-06-15,Approved
5,2024-12-19,Pending
6,2019-05-22,Rejected
7,2019-03-25,Approved
8,2021-04-30,Pending
9,2022-01-19,Approved


In [53]:
loan_df.loc[
    loan_df["loan_status"] == "Rejected",
    "loan_start_date"
] = None

In [58]:
loan_df[
    loan_df["loan_status"] == "Rejected"
][["loan_status", "loan_start_date"]].head()

,loan_status,loan_start_date
0,Rejected,None
2,Rejected,None
6,Rejected,None
12,Rejected,None
15,Rejected,None


In [59]:
loan_df["loan_start_date"].isnull().sum()

np.int64(19530)

In [60]:
loan_df["loan_start_date"] = [
    generate_loan_start_date()
    for _ in range(len(loan_df))
]

In [61]:
loan_df["loan_start_date"].isnull().sum()

np.int64(0)

In [62]:
loans_final = loan_df[
    [
        "customer_id",
        "loan_type",
        "loan_amount",
        "interest_rate",
        "tenure_months",
        "emi",
        "loan_status",
        "loan_start_date"
    ]
]

loans_final.to_sql(
    "loans",
    con=engine,
    if_exists="append",
    index=False
)

100000

In [63]:
SELECT COUNT(*) FROM loans;

SyntaxError: Invalid star expression (1616062738.py, line 1)

In [66]:
df = pd.read_sql(text("""SELECT COUNT(*) FROM loans;""") ,engine)

In [67]:
df

,COUNT(*)
0,100000
